# Data Setup Guide

This notebook walks through downloading and preparing the three datasets
for Fed-Vis. Each dataset represents one hospital site in our federation.

| Dataset | Organ | Sites | Source |
|---------|-------|-------|--------|
| FeTS 2022 | Brain | 4 sites | Synapse |
| Prostate | Prostate | 6 sites | NCI-ISBI / Zenodo |
| CT Lung | Lung | 5 sites | Kaggle / TCIA |

## 1. Directory Structure

All data goes under a single root directory. Set this to wherever you have space.

In [ ]:
import os
from pathlib import Path

# change this to your data location
DATA_ROOT = Path("../data")

# create the expected folder structure
for name in ["FeTS2022", "Prostate_processed", "CTLung"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

print("Data root:", DATA_ROOT.resolve())

## 2. FeTS 2022 (Brain Tumors)

**What:** Multi-modal brain MRI (T1, T1ce, T2, FLAIR) with tumor masks.  
**Sites:** 4 institutions (labeled 1, 6, 18, 21).  
**Format:** NIfTI (`.nii.gz`).

### Download

1. Go to [Synapse - FeTS 2022](https://www.synapse.org/#!Synapse:syn28546456)
2. Create an account and accept the data use agreement
3. Download the training data partition
4. Extract into `data/FeTS2022/`

Expected structure after extraction:
```
data/FeTS2022/
├── 1/                     # site 1
│   └── images/
│       ├── BraTS-GLI-00001.nii.gz
│       └── ...
│   └── labels/
│       ├── BraTS-GLI-00001.nii.gz
│       └── ...
├── 6/                     # site 6
├── 18/                    # site 18
└── 21/                    # site 21
```

In [ ]:
# verify FeTS data
fets_path = DATA_ROOT / "FeTS2022"
for site in ['1', '6', '18', '21']:
    site_dir = fets_path / site / 'images'
    if site_dir.exists():
        n = len(list(site_dir.glob('*.nii.gz')))
        print(f"  Site {site}: {n} volumes")
    else:
        print(f"  Site {site}: NOT FOUND - download from Synapse")

## 3. Multi-Site Prostate MRI

**What:** T2-weighted prostate MRI with gland segmentation masks.  
**Sites:** BIDMC, HK, I2CVB, ISBI, ISBI_1.5, UCL.  
**Format:** NIfTI (`.nii.gz`).

### Download

1. Go to [NCI-ISBI 2013 Prostate Challenge](https://wiki.cancerimagingarchive.net/display/Public/NCI-ISBI+2013+Challenge+-+Automated+Segmentation+of+Prostate+Structures) or search on Zenodo
2. Download the multi-site partition
3. Extract into `data/Prostate_processed/`

Expected structure:
```
data/Prostate_processed/
├── BIDMC/
│   ├── images/
│   │   ├── Case00.nii.gz
│   │   └── ...
│   └── labels/
│       ├── Case00.nii.gz
│       └── ...
├── HK/
├── I2CVB/
├── ISBI/
├── ISBI_1.5/
└── UCL/
```

In [ ]:
# verify Prostate data
prostate_path = DATA_ROOT / "Prostate_processed"
for site in ['BIDMC', 'HK', 'I2CVB', 'ISBI', 'ISBI_1.5', 'UCL']:
    site_dir = prostate_path / site / 'images'
    if site_dir.exists():
        n = len(list(site_dir.glob('*.nii.gz')))
        print(f"  {site}: {n} volumes")
    else:
        print(f"  {site}: NOT FOUND")

## 4. CT Lung

**What:** Chest CT scans with lung nodule/tumor segmentation.  
**Sites:** 5 institutions (labeled 1-5).  
**Format:** PNG images with masks.

### Download

1. Search for "COVID-19 CT Lung and Infection Segmentation" on Kaggle or TCIA
2. Download and extract into `data/CTLung/`

Expected structure:
```
data/CTLung/
├── 1/
│   ├── images/
│   │   ├── slice_001.png
│   │   └── ...
│   └── masks/
│       ├── slice_001.png
│       └── ...
├── 2/
├── 3/
├── 4/
└── 5/
```

In [ ]:
# verify CT Lung data
lung_path = DATA_ROOT / "CTLung"
for site in ['1', '2', '3', '4', '5']:
    site_dir = lung_path / site / 'images'
    if site_dir.exists():
        n = len(list(site_dir.glob('*')))
        print(f"  Site {site}: {n} slices")
    else:
        print(f"  Site {site}: NOT FOUND")

## 5. Preprocessing

Once the raw data is downloaded, our loaders handle preprocessing on the fly:
- Z-score normalization (MRI) or windowing (CT)
- Crop/pad volumes to 64×128×128
- Binary label mapping (any tumor → class 1)

No separate preprocessing script needed — it happens inside the dataset classes.

In [ ]:
# quick test: try loading one sample if data is available
import sys
sys.path.insert(0, '../src')

from omegaconf import OmegaConf

cfg = OmegaConf.create({
    'data': {
        'processed_path': str(fets_path),
        'volume_size': {'depth': 64, 'height': 128, 'width': 128},
        'train_ratio': 0.8,
        'val_ratio': 0.1,
        'test_ratio': 0.1,
        'sites': ['1', '6', '18', '21'],
        'use_modality': 'T1',
        'binary_labels': True,
    },
    'paths': {'data_root': str(DATA_ROOT)},
})

try:
    from fedvis.data import FeTSDataset
    ds = FeTSDataset(cfg, split='train', site='1')
    print(f"Loaded {len(ds)} training samples from FeTS site 1")
    
    if len(ds) > 0:
        vol, mask = ds[0]
        print(f"Volume shape: {vol.shape}")
        print(f"Mask shape:   {mask.shape}")
        print(f"Value range:  [{vol.min():.2f}, {vol.max():.2f}]")
except Exception as e:
    print(f"Could not load data: {e}")
    print("This is expected if datasets haven't been downloaded yet.")

## 6. Summary

| Dataset | Download From | Expected Location |
|---------|--------------|-------------------|
| FeTS 2022 | Synapse (needs account + DUA) | `data/FeTS2022/{site}/images/*.nii.gz` |
| Prostate | NCI-ISBI / Zenodo | `data/Prostate_processed/{site}/images/*.nii.gz` |
| CT Lung | Kaggle / TCIA | `data/CTLung/{site}/images/*.png` |

Once the raw data is in place, proceed to **`02_training.ipynb`** to train the model.